# 37 — Scheduling Specifies Efficient Distribution

**Leading specification:** discovered candidate replicas become useful when transfers are ordered into a verifiable schedule.

Notebook 29 established that discovery maps a content identity to candidate locations:

\[
D(I)\rightarrow\{L_1,L_2,\ldots,L_n\}
\]

Notebook 37 asks the next systems question:

> Once candidate sources exist, which transfer should happen first?

Scheduling answers by turning candidate chunk-source pairs into an ordered transfer plan. The scheduler may optimize latency, load, or failure risk, but every received chunk still verifies against its content identity.

## 1. Context

Open model distribution now has:

```text
identity
→ verification
→ chunking
→ replication
→ discovery
```

Scheduling adds transfer order:

```text
candidate chunks
→ replica options
→ cost model
→ scheduled transfers
→ verify chunks
→ reassemble
```

Scheduling is not trust. It chooses an efficient order. Verification still decides whether bytes are accepted.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import shutil
import sys
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Robust paths whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "37_scheduling"
SITE = REPO_ROOT / "site"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
except Exception:
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")

## 2. Scheduling variables

A transfer queue contains candidate chunk-source pairs:

\[
Q=\{(c_i,L_j)\}
\]

A scheduler selects an order \(s\):

\[
s^\*=\arg\min_s \operatorname{cost}(s)
\]

This notebook uses a simple weighted cost:

\[
\operatorname{cost}(c_i,L_j)
=
\alpha\,\text{latency}_{j}
+
\beta\,\text{load}_{j}
+
\gamma\,\text{risk}_{j}
\]

After transfer, verification remains mandatory:

\[
V_i=\bigl(H(c_i^{\mathrm{received}})=I_i\bigr)
\]

In [ ]:
@dataclass(frozen=True)
class SchedulingVariables:
    queue: str
    cost: str
    schedule: str
    verification: str
    objective: str

variables = SchedulingVariables(
    queue="Q={(c_i,L_j)} candidate chunk-source transfers",
    cost="weighted latency, load, and failure-risk penalty",
    schedule="ordered transfer plan selected from Q",
    verification="V_i checks received chunk bytes against chunk identity",
    objective="efficient retrieval without weakening verification",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})

## 3. Create chunks and candidate replicas

The notebook creates four chunks and three possible replica sources. Each source has latency, load, and failure-risk metadata.

In [ ]:
artifact_dir = RESULTS / "artifacts"
source_root = RESULTS / "sources"
received_dir = RESULTS / "received"
for d in [artifact_dir, source_root, received_dir]:
    d.mkdir(parents=True, exist_ok=True)

chunk_payloads = []
for idx in range(4):
    payload = (
        f"open-model-distribution scheduling notebook chunk {idx}\n".encode()
        + bytes([(idx + k) % 256 for k in range(1024)])
    )
    chunk_payloads.append(payload)

chunks = []
for idx, payload in enumerate(chunk_payloads):
    path = artifact_dir / f"chunk_{idx:04d}.bin"
    path.write_bytes(payload)
    digest = sha256_file(path)
    chunks.append({
        "chunk_index": idx,
        "path": str(path.relative_to(REPO_ROOT)),
        "size_bytes": path.stat().st_size,
        "sha256": digest,
        "content_id": f"sha256:{digest}",
    })

chunk_df = pd.DataFrame(chunks)
chunk_table_csv = RESULTS / "37_chunk_table.csv"
chunk_df.to_csv(chunk_table_csv, index=False)

sources = pd.DataFrame([
    {"source": "server_A", "kind": "server", "latency_ms": 45, "load": 0.70, "failure_risk": 0.10},
    {"source": "mirror_B", "kind": "mirror", "latency_ms": 85, "load": 0.25, "failure_risk": 0.08},
    {"source": "peer_C", "kind": "peer", "latency_ms": 60, "load": 0.40, "failure_risk": 0.22},
])

source_costs_csv = RESULTS / "37_replica_costs.csv"
sources.to_csv(source_costs_csv, index=False)

chunk_df, sources

In [ ]:
# Replicate chunks to each source. Make one source/chunk invalid to preserve the verification lesson.
for _, src in sources.iterrows():
    sdir = source_root / src["source"]
    sdir.mkdir(parents=True, exist_ok=True)
    for _, chunk in chunk_df.iterrows():
        source_path = REPO_ROOT / chunk["path"]
        dst = sdir / f"chunk_{int(chunk['chunk_index']):04d}.bin"
        shutil.copyfile(source_path, dst)

# Inject one invalid candidate: peer_C chunk 2.
bad_path = source_root / "peer_C" / "chunk_0002.bin"
bad = bytearray(bad_path.read_bytes())
bad[100] = (bad[100] + 1) % 256
bad_path.write_bytes(bytes(bad))

print("Created candidate sources with one invalid candidate.")

## 4. Build the transfer queue

The queue contains every candidate pair \((c_i,L_j)\).

In [ ]:
alpha = 1.0
beta = 60.0
gamma = 120.0

queue_rows = []
for _, chunk in chunk_df.iterrows():
    for _, src in sources.iterrows():
        candidate_path = source_root / src["source"] / f"chunk_{int(chunk['chunk_index']):04d}.bin"
        queue_rows.append({
            "chunk_index": int(chunk["chunk_index"]),
            "chunk_identity": chunk["content_id"],
            "expected_sha256": chunk["sha256"],
            "source": src["source"],
            "kind": src["kind"],
            "latency_ms": src["latency_ms"],
            "load": src["load"],
            "failure_risk": src["failure_risk"],
            "candidate_path": str(candidate_path.relative_to(REPO_ROOT)),
            "cost": alpha * src["latency_ms"] + beta * src["load"] + gamma * src["failure_risk"],
        })

queue_df = pd.DataFrame(queue_rows)
transfer_queue_csv = RESULTS / "37_transfer_queue.csv"
queue_df.to_csv(transfer_queue_csv, index=False)

queue_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axis("off")
ax.set_title("Scheduling pipeline", fontsize=17, pad=16)

nodes = [
    ("candidate\nchunks", 0.10),
    ("replica\noptions", 0.29),
    ("cost\nmodel", 0.47),
    ("scheduled\ntransfers", 0.67),
    ("verify\nchunks", 0.87),
]

for label, x in nodes:
    ax.text(x, 0.58, label, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.3),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(x2 - 0.06, 0.58), xytext=(x1 + 0.06, 0.58),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.20, "scheduling optimizes order; verification still decides acceptance", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
scheduling_pipeline_fig = FIGURES / "37_scheduling_pipeline.png"
fig.savefig(scheduling_pipeline_fig, dpi=180)
plt.show()

scheduling_pipeline_fig

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))

for source, sub in queue_df.groupby("source"):
    ax.scatter(sub["chunk_index"], sub["cost"], label=source)

ax.set_xlabel("chunk index")
ax.set_ylabel("transfer cost")
ax.set_title("Candidate transfer costs")
ax.set_xticks(sorted(queue_df["chunk_index"].unique()))
ax.legend()
fig.tight_layout()

replica_costs_fig = FIGURES / "37_replica_costs.png"
fig.savefig(replica_costs_fig, dpi=180)
plt.show()

replica_costs_fig

## 5. Build a schedule

For each chunk, select the lowest-cost candidate first. If verification fails, the client falls back to the next candidate.

In [ ]:
schedule_rows = []
for chunk_index, candidates in queue_df.groupby("chunk_index"):
    ordered = candidates.sort_values("cost").reset_index(drop=True)
    for rank, row in ordered.iterrows():
        schedule_rows.append({
            "chunk_index": int(chunk_index),
            "attempt_rank": int(rank + 1),
            "source": row["source"],
            "candidate_path": row["candidate_path"],
            "cost": row["cost"],
            "expected_sha256": row["expected_sha256"],
        })

schedule_df = pd.DataFrame(schedule_rows)
schedule_csv = RESULTS / "37_schedule.csv"
schedule_df.to_csv(schedule_csv, index=False)

schedule_df.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
first_choices = schedule_df[schedule_df["attempt_rank"] == 1].copy()
ax.bar(first_choices["chunk_index"], first_choices["cost"])
ax.set_xlabel("chunk index")
ax.set_ylabel("selected first-attempt cost")
ax.set_title("Scheduled first attempts")
ax.set_xticks(first_choices["chunk_index"])

for idx, row in first_choices.iterrows():
    ax.text(row["chunk_index"], row["cost"] + 3, row["source"], ha="center", va="bottom", fontsize=9)

fig.tight_layout()
scheduled_downloads_fig = FIGURES / "37_scheduled_downloads.png"
fig.savefig(scheduled_downloads_fig, dpi=180)
plt.show()

scheduled_downloads_fig

## 6. Execute the schedule with verification

Scheduling chooses candidates. Verification accepts bytes.

If a scheduled candidate fails verification, the next candidate for that chunk is attempted.

In [ ]:
accepted_chunks = {}
attempt_records = []

for chunk_index, candidates in schedule_df.groupby("chunk_index"):
    for _, candidate in candidates.sort_values("attempt_rank").iterrows():
        src = REPO_ROOT / candidate["candidate_path"]
        dst = received_dir / f"chunk_{int(chunk_index):04d}_from_{candidate['source']}.bin"
        shutil.copyfile(src, dst)
        local_digest = sha256_file(dst)
        passed = local_digest == candidate["expected_sha256"]

        record = {
            "chunk_index": int(chunk_index),
            "attempt_rank": int(candidate["attempt_rank"]),
            "source": candidate["source"],
            "cost": candidate["cost"],
            "retrieved_path": str(dst.relative_to(REPO_ROOT)),
            "local_sha256": local_digest,
            "expected_sha256": candidate["expected_sha256"],
            "verification": "PASS" if passed else "FAIL",
            "accepted": passed,
        }
        attempt_records.append(record)

        if passed:
            accepted_chunks[int(chunk_index)] = record
            break

attempts_df = pd.DataFrame(attempt_records)
attempts_csv = RESULTS / "37_transfer_attempts.csv"
attempts_df.to_csv(attempts_csv, index=False)

accepted_df = pd.DataFrame(list(accepted_chunks.values())).sort_values("chunk_index")
accepted_csv = RESULTS / "37_accepted_transfers.csv"
accepted_df.to_csv(accepted_csv, index=False)

attempts_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axis("off")
ax.set_title("Verify after scheduling", fontsize=17, pad=16)

for idx, row in accepted_df.iterrows():
    x = 0.15 + int(row["chunk_index"]) * 0.20
    ax.text(x, 0.60, f"chunk {int(row['chunk_index'])}\n{row['source']}", ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)
    ax.text(x, 0.36, "PASS", ha="center", fontsize=12, transform=ax.transAxes)

ax.text(0.5, 0.15, "schedule order is provisional; digest verification is final", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
verify_after_schedule_fig = FIGURES / "37_verify_after_schedule.png"
fig.savefig(verify_after_schedule_fig, dpi=180)
plt.show()

verify_after_schedule_fig

## 7. Scheduling report

The report records the queue, selected attempts, fallbacks, and verification results.

In [ ]:
fallback_count = int((attempts_df["attempt_rank"] > 1).sum())
accepted_count = int(accepted_df["accepted"].sum())

verification_report = {
    "schema": "open-model-distribution.scheduling-report.v0",
    "chunk_count": int(chunk_df.shape[0]),
    "candidate_count": int(queue_df.shape[0]),
    "accepted_count": accepted_count,
    "fallback_attempts": fallback_count,
    "all_chunks_verified": accepted_count == int(chunk_df.shape[0]),
    "statement": "Scheduling optimizes transfer order; verification still accepts chunks.",
    "accepted_transfers": accepted_df.to_dict(orient="records"),
    "attempts": attempts_df.to_dict(orient="records"),
}

verification_report_path = RESULTS / "37_verification_report.json"
verification_report_path.write_text(json.dumps(verification_report, indent=2), encoding="utf-8")

verification_report

## 8. Engineering summary

| Property | Specification | Notebook role |
|---|---|---|
| Queue | \(Q=\{(c_i,L_j)\}\) | Candidate chunk-source transfers. |
| Cost | latency + load + risk | Orders candidate sources. |
| Schedule | ordered attempts | Chooses what to try first. |
| Fallback | next candidate | Handles failed verification or transfer failure. |
| Verification | \(V_i\) | Accepts only chunks matching content identity. |

## 9. Engineering statement

Scheduling specifies efficient distribution by turning discovered candidate replicas into an ordered transfer plan. The scheduler may optimize latency, load, or failure risk, but every received chunk still verifies against its content identity.

## 10. Key equations

Transfer queue:

\[
Q=\{(c_i,L_j)\}
\]

Candidate cost:

\[
\operatorname{cost}(c_i,L_j)
=
\alpha\,\mathrm{latency}_{j}
+
\beta\,\mathrm{load}_{j}
+
\gamma\,\mathrm{risk}_{j}
\]

Scheduling objective:

\[
s^\*=\arg\min_s \operatorname{cost}(s)
\]

Verification condition:

\[
V_i=\bigl(H(c_i^{\mathrm{received}})=I_i\bigr)
\]

Scheduling principle:

\[
\text{efficient distribution}
=
\text{ordered transfers}
+
\text{local verification}.
\]

## 11. Notebook outputs

Notebook 37 writes:

- `results/37_scheduling/37_chunk_table.csv`
- `results/37_scheduling/37_replica_costs.csv`
- `results/37_scheduling/37_transfer_queue.csv`
- `results/37_scheduling/37_schedule.csv`
- `results/37_scheduling/37_transfer_attempts.csv`
- `results/37_scheduling/37_accepted_transfers.csv`
- `results/37_scheduling/37_verification_report.json`
- `figures/37_scheduling_pipeline.png`
- `figures/37_replica_costs.png`
- `figures/37_scheduled_downloads.png`
- `figures/37_verify_after_schedule.png`

In [ ]:
notebook_manifest = {
    "notebook": "37_scheduling.ipynb",
    "title": "Scheduling Specifies Efficient Distribution",
    "primary_specification": "scheduling",
    "statement": "Discovered candidate replicas become useful when transfers are ordered into a verifiable schedule.",
    "equations": [
        "Q={(c_i,L_j)}",
        "cost=alpha latency + beta load + gamma risk",
        "s*=argmin cost(s)",
        "V_i=(H(c_i_received)=I_i)",
        "efficient distribution=ordered transfers+local verification",
    ],
    "outputs": {
        "chunk_table_csv": str(chunk_table_csv.relative_to(REPO_ROOT)),
        "replica_costs_csv": str(source_costs_csv.relative_to(REPO_ROOT)),
        "transfer_queue_csv": str(transfer_queue_csv.relative_to(REPO_ROOT)),
        "schedule_csv": str(schedule_csv.relative_to(REPO_ROOT)),
        "transfer_attempts_csv": str(attempts_csv.relative_to(REPO_ROOT)),
        "accepted_transfers_csv": str(accepted_csv.relative_to(REPO_ROOT)),
        "verification_report": str(verification_report_path.relative_to(REPO_ROOT)),
        "scheduling_pipeline_figure": str(scheduling_pipeline_fig.relative_to(REPO_ROOT)),
        "replica_costs_figure": str(replica_costs_fig.relative_to(REPO_ROOT)),
        "scheduled_downloads_figure": str(scheduled_downloads_fig.relative_to(REPO_ROOT)),
        "verify_after_schedule_figure": str(verify_after_schedule_fig.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "37_notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

notebook_manifest

## 12. Download artifacts

Run the final cell to package notebook 37 outputs for download.

In [ ]:
# Package notebook artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

NOTEBOOKS = REPO_ROOT / "notebooks"
SITE = REPO_ROOT / "site"

zip_path = RESULTS / "37_scheduling_artifacts.zip"

notebook_path = NOTEBOOKS / "37_scheduling.ipynb"
fallback_notebook_path = Path.cwd() / "37_scheduling.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)

        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)

        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass